In [1]:
import os, sys
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['XLA_PYTHON_CLIENT_ALLOCATOR'] = 'platform'
os.environ['JAX_PLATFORM_NAME'] = 'cpu'  # change to 'gpu' for GPU runs

from jax import config
config.update('jax_enable_x64', True)

sys.path.insert(0, os.path.abspath('..'))

# OTT-JAX integration benchmark

This notebook shows how to run OTT-JAX solvers side-by-side with uot-bench native solvers inside the same `Generator → Problem → BaseSolver → Experiment → run_pipeline → DataFrame` pipeline.

**Sections**
1. [Sanity check](#1-sanity-check) — imports and a single-problem smoke test
2. [Sinkhorn comparison](#2-sinkhorn-comparison) — native `SinkhornTwoMarginalLogJaxSolver` vs `OTTSinkhornSolver` across epsilon grid
3. [Low-rank Sinkhorn](#3-low-rank-sinkhorn) — `OTTLRSinkhornSolver` across rank × epsilon grid
4. [Gromov–Wasserstein](#4-gromov-wasserstein) — `OTTGromovWassersteinSolver` (new capability in uot-bench)
5. [Sinkhorn divergence](#5-sinkhorn-divergence) — `OTTSinkhornDivergence`
6. [Full comparison table & plots](#6-full-comparison)

**Requirements**: `pip install uot-bench[ott]`

In [2]:
import time
import math
import logging

import numpy as np
import pandas as pd
import jax.numpy as jnp
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- uot-bench core ---
from uot.problems.generators import GaussianMixtureGenerator
from uot.problems.iterator import OnlineProblemIterator
from uot.solvers.solver_config import SolverConfig
from uot.experiments.runner import run_pipeline
from uot.experiments.experiment import Experiment
from uot.experiments.measurement import measure_time_and_output
from uot.utils.costs import cost_euclid_squared

# --- native uot-bench solvers ---
from uot.solvers.sinkhorn import SinkhornTwoMarginalLogJaxSolver

# --- OTT-JAX adapter ---
from uot.interop.ott import (
    OTTSinkhornSolver,
    OTTLRSinkhornSolver,
    OTTGromovWassersteinSolver,
    OTTSinkhornDivergence,
)

# Quiet the pipeline's per-solve INFO logging in the notebook (keep warnings/errors).
# Must run after importing uot, which sets the 'uot' logger to INFO at import time.
logging.getLogger('uot').setLevel(logging.WARNING)

print('All imports OK.')

All imports OK.


## 1  Sanity check

Build one small problem and call each solver directly — no experiment harness yet.

In [3]:
from uot.data.measure import PointCloudMeasure
from uot.problems.two_marginal import TwoMarginalProblem

rng = np.random.default_rng(42)
n = 30
mu = PointCloudMeasure(
    jnp.asarray(rng.standard_normal((n, 2)).astype(np.float32)),
    jnp.ones(n, dtype=jnp.float32) / n,
)
nu = PointCloudMeasure(
    jnp.asarray(rng.standard_normal((n, 2)).astype(np.float32) + 2.0),
    jnp.ones(n, dtype=jnp.float32) / n,
)
toy = TwoMarginalProblem('sanity', mu, nu, cost_euclid_squared)

marginals = toy.get_marginals()
costs     = toy.get_costs()

eps = 0.1

# OTT linear/quadratic wrappers consume a *pre-built* OTT problem object
# (their input_kind is "ott_linear" / "ott_quadratic"), so build it first.
lin_prob_eps  = toy.to_ott_linear_problem(epsilon=eps)
quad_prob_eps = toy.to_ott_quadratic_problem(epsilon=eps)

# Native Sinkhorn (takes raw marginals/costs)
native_out = SinkhornTwoMarginalLogJaxSolver().solve(marginals, costs, reg=eps, maxiter=1000)
print(f'Native Sinkhorn    cost={float(native_out["cost"]):.6f}  converged={native_out.get("converged")}')

# OTT Sinkhorn
ott_out = OTTSinkhornSolver(max_iterations=2000, threshold=1e-5).solve(lin_prob_eps, epsilon=eps)
print(f'OTT Sinkhorn       cost={float(ott_out["cost"]):.6f}  converged={ott_out.get("converged")}')

# OTT LR Sinkhorn
lr_out = OTTLRSinkhornSolver(rank=8).solve(lin_prob_eps, epsilon=eps)
print(f'OTT LR-Sinkhorn    cost={float(lr_out["cost"]):.6f}  converged={lr_out.get("converged")}  low_rank_plan shapes={(lr_out["low_rank_plan"][0].shape, lr_out["low_rank_plan"][1].shape)}')

# OTT GW
gw_out = OTTGromovWassersteinSolver(epsilon=eps).solve(quad_prob_eps)
print(f'OTT GW             cost={float(gw_out["cost"]):.6f}  converged={gw_out.get("converged")}')

# OTT Sinkhorn divergence (input_kind="marginals_costs" — takes raw marginals/costs)
div_out = OTTSinkhornDivergence().solve(marginals, costs, epsilon=eps)
print(f'OTT SD             S_eps={float(div_out["cost"]):.6f}  converged={div_out.get("converged")}')

Native Sinkhorn    cost=6.321225  converged=None
OTT Sinkhorn       cost=6.321604  converged=False
OTT LR-Sinkhorn    cost=8.495642  converged=True  low_rank_plan shapes=((30, 8), (30, 8))
OTT GW             cost=2.184744  converged=True
OTT SD             S_eps=6.258297  converged=True


### Visualise the transport plan

A quick visual sanity check: the source `μ` and target `ν` point clouds with the OTT Sinkhorn coupling drawn as edges (only entries carrying significant mass are shown). The cost figures above are only convincing if the plan actually moves `μ` onto `ν`.

In [4]:
# Source/target clouds + OTT Sinkhorn transport plan.
mu_pts, _ = marginals[0].as_point_cloud()
nu_pts, _ = marginals[1].as_point_cloud()
mu_pts, nu_pts = np.asarray(mu_pts), np.asarray(nu_pts)
P = np.asarray(ott_out['transport_plan'])

# Draw only edges that carry non-trivial mass; pack into one trace via None separators.
significant = P > 1e-2 * P.max()
edge_x, edge_y = [], []
for i, j in zip(*np.where(significant)):
    edge_x += [mu_pts[i, 0], nu_pts[j, 0], None]
    edge_y += [mu_pts[i, 1], nu_pts[j, 1], None]

fig_toy = go.Figure()
fig_toy.add_trace(go.Scatter(x=edge_x, y=edge_y, mode='lines',
                             line=dict(color='rgba(120,120,120,0.4)', width=1),
                             hoverinfo='skip', name='transport'))
fig_toy.add_trace(go.Scatter(x=mu_pts[:, 0], y=mu_pts[:, 1], mode='markers',
                             marker=dict(color='royalblue', size=9), name='source μ'))
fig_toy.add_trace(go.Scatter(x=nu_pts[:, 0], y=nu_pts[:, 1], mode='markers',
                             marker=dict(color='crimson', size=9), name='target ν'))
fig_toy.update_layout(height=500, width=600,
                      title=f'OTT Sinkhorn transport plan (ε={eps}) — {int(significant.sum())} significant edges',
                      xaxis_title='x₀', yaxis_title='x₁')
fig_toy.show()

The Gromov–Wasserstein plan couples points by **intra-domain** structure (pairwise distances within each cloud), not by absolute position, so its edges need not look like a clean spatial translation the way the Sinkhorn plan does.

In [5]:
# Source/target clouds + OTT Gromov–Wasserstein transport plan.
P_gw = np.asarray(gw_out['transport_plan'])

significant_gw = P_gw > 1e-2 * P_gw.max()
edge_x_gw, edge_y_gw = [], []
for i, j in zip(*np.where(significant_gw)):
    edge_x_gw += [mu_pts[i, 0], nu_pts[j, 0], None]
    edge_y_gw += [mu_pts[i, 1], nu_pts[j, 1], None]

fig_gw_plan = go.Figure()
fig_gw_plan.add_trace(go.Scatter(x=edge_x_gw, y=edge_y_gw, mode='lines',
                                 line=dict(color='rgba(255,140,0,0.4)', width=1),
                                 hoverinfo='skip', name='transport'))
fig_gw_plan.add_trace(go.Scatter(x=mu_pts[:, 0], y=mu_pts[:, 1], mode='markers',
                                 marker=dict(color='royalblue', size=9), name='source μ'))
fig_gw_plan.add_trace(go.Scatter(x=nu_pts[:, 0], y=nu_pts[:, 1], mode='markers',
                                 marker=dict(color='crimson', size=9), name='target ν'))
fig_gw_plan.update_layout(height=500, width=600,
                          title=f'OTT Gromov–Wasserstein transport plan (ε={eps}) — {int(significant_gw.sum())} significant edges',
                          xaxis_title='x₀', yaxis_title='x₁')
fig_gw_plan.show()

### 1a  Inspect OTT output fields and the OTT LinearProblem helper

In [6]:
# Convert directly to an OTT LinearProblem (useful for manual OTT usage)
lin_prob = toy.to_ott_linear_problem(epsilon=eps, tau_a=1.0, tau_b=1.0)
print('OTT LinearProblem:', lin_prob)
print('  geom type:', type(lin_prob.geom).__name__)
print('  a shape:', lin_prob.a.shape, '  b shape:', lin_prob.b.shape)

print()
print('OTT Sinkhorn output fields:', list(ott_out.keys()))
print('OTT LR output fields:', list(lr_out.keys()))
print('GW output fields:', list(gw_out.keys()))

OTT LinearProblem: <ott.problems.linear.linear_problem.LinearProblem object at 0x34ba1f000>
  geom type: PointCloud
  a shape: (30,)   b shape: (30,)

OTT Sinkhorn output fields: ['cost', 'transport_plan', 'u_final', 'v_final', 'potentials', 'iterations', 'converged', 'error']
OTT LR output fields: ['cost', 'low_rank_plan', 'iterations', 'converged', 'error']
GW output fields: ['cost', 'iterations', 'converged', 'error', 'transport_plan']


## 2  Sinkhorn comparison

Run native `SinkhornTwoMarginalLogJaxSolver` vs `OTTSinkhornSolver` across an `epsilon` grid on 10 Gaussian-mixture problems. The results land in the same `pandas.DataFrame`.

In [7]:
N_PROBLEMS = 10   # number of OT instances per fold
N_POINTS   = 50   # points per measure
DIM        = 2    # ambient dimension
SEED       = 7

gen_cfg = dict(
    name=f'GMM-{DIM}d-{N_POINTS}pt',
    dim=DIM,
    num_components=2,
    n_points=N_POINTS,
    num_datasets=N_PROBLEMS,
    borders=(0, 1),
    cost_fn=cost_euclid_squared,
    use_jax=True,
    seed=SEED,
    measure_mode='point_cloud',
)

def make_iterator():
    return OnlineProblemIterator(GaussianMixtureGenerator(**gen_cfg), num=N_PROBLEMS, cache_gt=False)

experiment = Experiment(
    name='OTT vs Native Sinkhorn',
    solve_fn=measure_time_and_output,
)

eps_grid = [1.0, 0.1, 0.01]

# NOTE: SolverConfig.solver takes a class (not an instance).
# Both __init__ and solve() kwargs live in param_grid;
# instantiate_solver filters the right subset for __init__.
solvers_sinkhorn = [
    SolverConfig(
        name='native-sinkhorn-log',
        solver=SinkhornTwoMarginalLogJaxSolver,
        param_grid=[{'reg': eps, 'maxiter': 2000} for eps in eps_grid],
        is_jit=False,
        use_cost_matrix=True,
    ),
    SolverConfig(
        name='ott-sinkhorn',
        solver=OTTSinkhornSolver,
        param_grid=[{'epsilon': eps, 'max_iterations': 2000, 'threshold': 1e-5} for eps in eps_grid],
        is_jit=False,
        use_cost_matrix=False,
    ),
]

results_sinkhorn = run_pipeline(
    experiment=experiment,
    solvers=solvers_sinkhorn,
    iterators=[make_iterator()],
    folds=1,
    progress=False,
)

print('Result shape:', results_sinkhorn.shape)
results_sinkhorn[['name', 'cost', 'iterations', 'converged', 'time', 'error']].head(12)

# Consistent colours across all figures — keyed by solver name.
import plotly.express as px
_pal = px.colors.qualitative.Plotly
SOLVER_COLORS = {
    'native-sinkhorn-log':      _pal[0],   # blue
    'ott-sinkhorn':             _pal[1],   # red
    'ott-lr-sinkhorn':         _pal[2],   # green
    'ott-sinkhorn-ref':         _pal[3],   # purple
    'ott-gw':                   _pal[4],   # orange
    'ott-sinkhorn-divergence':  _pal[5],   # cyan
}
def solver_color(name: str) -> str:
    return SOLVER_COLORS.get(name, '#888888')

2026-06-15 17:09:29,115 uot.problems.iterator INFO: Generated problem <TwoMarginalProblem[GMM-2d-50pt] 5000x5000 with (cost_euclid_squared)>
2026-06-15 17:09:29,115 uot.problems.iterator INFO: Generated problem <TwoMarginalProblem[GMM-2d-50pt] 5000x5000 with (cost_euclid_squared)>
2026-06-15 17:09:30,096 uot.problems.iterator INFO: Generated problem <TwoMarginalProblem[GMM-2d-50pt] 5000x5000 with (cost_euclid_squared)>
2026-06-15 17:09:30,096 uot.problems.iterator INFO: Generated problem <TwoMarginalProblem[GMM-2d-50pt] 5000x5000 with (cost_euclid_squared)>
2026-06-15 17:09:31,023 uot.problems.iterator INFO: Generated problem <TwoMarginalProblem[GMM-2d-50pt] 5000x5000 with (cost_euclid_squared)>
2026-06-15 17:09:31,023 uot.problems.iterator INFO: Generated problem <TwoMarginalProblem[GMM-2d-50pt] 5000x5000 with (cost_euclid_squared)>
2026-06-15 17:09:31,968 uot.problems.iterator INFO: Generated problem <TwoMarginalProblem[GMM-2d-50pt] 5000x5000 with (cost_euclid_squared)>
2026-06-15 17

In [8]:
sinkhorn_ok = results_sinkhorn[results_sinkhorn['status'] == 'success'].copy()

for col in ['cost', 'time', 'iterations', 'error']:
    if col in sinkhorn_ok.columns:
        sinkhorn_ok[col] = sinkhorn_ok[col].apply(lambda v: float(v) if v is not None else float('nan'))

def get_eps(row):
    reg = row.get('reg')
    if reg is not None and not (isinstance(reg, float) and math.isnan(reg)):
        return float(reg)
    eps = row.get('epsilon')
    return float(eps) if eps is not None else float('nan')

sinkhorn_ok['eps'] = sinkhorn_ok.apply(get_eps, axis=1)
grouped = sinkhorn_ok.groupby(['name', 'eps'])[['cost', 'time', 'iterations']].median().reset_index()

fig = make_subplots(rows=1, cols=3, subplot_titles=['Median cost', 'Median wall time (ms)', 'Median iterations'])

for solver_name, grp in grouped.groupby('name'):
    grp = grp.sort_values('eps')
    color = solver_color(solver_name)
    common = dict(x=grp['eps'], name=solver_name, mode='lines+markers',
                  line_color=color, marker_color=color)
    fig.add_trace(go.Scatter(**common, y=grp['cost'],       showlegend=True),  row=1, col=1)
    fig.add_trace(go.Scatter(**common, y=grp['time'],       showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(**common, y=grp['iterations'], showlegend=False), row=1, col=3)

fig.update_xaxes(type='log', title_text='ε')
fig.update_layout(height=400, width=1000, title='Native Sinkhorn vs OTT Sinkhorn — median over problems')
fig.show()


## 3  Low-rank Sinkhorn

`OTTLRSinkhornSolver` stores the coupling as low-rank factors `(Q, R, g)` — the full `n×m` matrix is never materialised. Useful for larger problems.

In [9]:
rank_grid = [4, 8, 16]
eps_lr    = 0.1

# All rank variants in one SolverConfig; rank is an __init__ param so
# instantiate_solver creates a fresh OTTLRSinkhornSolver per param set.
solvers_lr = [
    SolverConfig(
        name='ott-lr-sinkhorn',
        solver=OTTLRSinkhornSolver,
        param_grid=[{'rank': r, 'gamma': 10.0, 'gamma_rescale': True, 'epsilon': eps_lr}
                    for r in rank_grid],
        is_jit=False,
        use_cost_matrix=False,
    ),
    # Full-rank OTT Sinkhorn reference at the same epsilon.
    SolverConfig(
        name='ott-sinkhorn-ref',
        solver=OTTSinkhornSolver,
        param_grid=[{'epsilon': eps_lr, 'max_iterations': 2000}],
        is_jit=False,
        use_cost_matrix=False,
    ),
]

results_lr = run_pipeline(
    experiment=experiment,
    solvers=solvers_lr,
    iterators=[make_iterator()],
    folds=1,
    progress=False,
)

lr_ok = results_lr[results_lr['status'] == 'success'].copy()
lr_ok['rank'] = lr_ok['rank'].apply(
    lambda v: int(v) if isinstance(v, float) and not math.isnan(v) else 'full'
)
lr_ok[['name', 'rank', 'cost', 'time', 'iterations', 'converged']].head(15)

2026-06-15 17:10:54,057 uot.problems.iterator INFO: Generated problem <TwoMarginalProblem[GMM-2d-50pt] 5000x5000 with (cost_euclid_squared)>
2026-06-15 17:10:54,057 uot.problems.iterator INFO: Generated problem <TwoMarginalProblem[GMM-2d-50pt] 5000x5000 with (cost_euclid_squared)>
2026-06-15 17:10:55,754 uot.problems.iterator INFO: Generated problem <TwoMarginalProblem[GMM-2d-50pt] 5000x5000 with (cost_euclid_squared)>
2026-06-15 17:10:55,754 uot.problems.iterator INFO: Generated problem <TwoMarginalProblem[GMM-2d-50pt] 5000x5000 with (cost_euclid_squared)>
2026-06-15 17:10:56,808 uot.problems.iterator INFO: Generated problem <TwoMarginalProblem[GMM-2d-50pt] 5000x5000 with (cost_euclid_squared)>
2026-06-15 17:10:56,808 uot.problems.iterator INFO: Generated problem <TwoMarginalProblem[GMM-2d-50pt] 5000x5000 with (cost_euclid_squared)>
2026-06-15 17:10:57,838 uot.problems.iterator INFO: Generated problem <TwoMarginalProblem[GMM-2d-50pt] 5000x5000 with (cost_euclid_squared)>
2026-06-15 17

,name,rank,cost,time,iterations,converged
0,ott-lr-sinkhorn,4,0.16706465988260366,1154.110875,50,True
1,ott-lr-sinkhorn,4,0.21136299841272266,517.057625,50,True
2,ott-lr-sinkhorn,4,0.0954393396039867,481.942459,70,True
3,ott-lr-sinkhorn,4,0.0743352127965785,833.191792,70,True
4,ott-lr-sinkhorn,4,0.0194040097597213,534.641500,30,True
5,ott-lr-sinkhorn,4,0.1454596191130051,291.896000,30,True
6,ott-lr-sinkhorn,4,0.011599423908052787,675.732416,50,True
7,ott-lr-sinkhorn,4,0.03237435064834084,523.410667,50,True
8,ott-lr-sinkhorn,4,0.12372470604450449,760.624458,50,True
9,ott-lr-sinkhorn,4,0.11895870680250112,419.860459,50,True


In [10]:
lr_rows  = lr_ok[lr_ok['rank'] != 'full'].copy()
ref_rows = lr_ok[lr_ok['rank'] == 'full'].copy()

for _df in [lr_rows, ref_rows]:
    for col in ['cost', 'time', 'iterations', 'error']:
        if col in _df.columns:
            _df[col] = _df[col].apply(lambda v: float(v) if v is not None else float('nan'))

all_ranks = sorted(lr_rows['rank'].unique())
all_ranks_str = [str(r) for r in all_ranks]

lr_median  = lr_rows.groupby('rank')[['cost', 'time']].median().reindex(all_ranks).reset_index()
lr_median['rank'] = lr_median['rank'].astype(str)

ref_median_cost = ref_rows['cost'].median()
ref_median_time = ref_rows['time'].median()

lr_color  = solver_color('ott-lr-sinkhorn')
ref_color = solver_color('ott-sinkhorn-ref')

fig_lr = make_subplots(rows=1, cols=2, subplot_titles=['Median cost by rank', 'Median wall time (ms)'])
fig_lr.add_trace(go.Bar(x=lr_median['rank'], y=lr_median['cost'],
                         name='ott-lr-sinkhorn', marker_color=lr_color), row=1, col=1)
fig_lr.add_trace(go.Bar(x=lr_median['rank'], y=lr_median['time'],
                         name='ott-lr-sinkhorn', marker_color=lr_color, showlegend=False), row=1, col=2)

for col_idx, ref_val in [(1, ref_median_cost), (2, ref_median_time)]:
    fig_lr.add_trace(go.Scatter(
        x=all_ranks_str, y=[ref_val] * len(all_ranks_str),
        mode='lines', line=dict(dash='dash', color=ref_color),
        name='ott-sinkhorn-ref', showlegend=(col_idx == 1),
    ), row=1, col=col_idx)

fig_lr.update_layout(
    barmode='group', height=400, width=900,
    title=f'Low-rank Sinkhorn (ε={eps_lr}) vs full-rank reference — median over problems',
)
fig_lr.show()
print('low_rank_plan factors are not serialised into the DataFrame (expected).')


low_rank_plan factors are not serialised into the DataFrame (expected).


## 4  Gromov–Wasserstein

`OTTGromovWassersteinSolver` is a **new capability** that did not exist in uot-bench before this adapter. It builds intra-space geometries from each measure's self-cost and solves the GW problem with an inner regularised Sinkhorn.

Note: the GW cost is scale-dependent and is not directly comparable to Sinkhorn costs.

In [11]:
eps_gw = [0.5, 0.1, 0.05]

solvers_gw = [
    SolverConfig(
        name='ott-gw',
        solver=OTTGromovWassersteinSolver,
        param_grid=[{
            'epsilon': eps,
            'linear_solver_lse_mode': True,
            'linear_solver_max_iterations': 500,
            'linear_solver_threshold': 1e-3,
        } for eps in eps_gw],
        is_jit=False,
        use_cost_matrix=False,
    ),
]

results_gw = run_pipeline(
    experiment=experiment,
    solvers=solvers_gw,
    iterators=[make_iterator()],
    folds=1,
    progress=False,
)

gw_ok = results_gw[results_gw['status'] == 'success'].copy()
gw_ok[['name', 'epsilon', 'cost', 'iterations', 'converged', 'time']].head(12)

2026-06-15 17:11:52,359 uot.problems.iterator INFO: Generated problem <TwoMarginalProblem[GMM-2d-50pt] 5000x5000 with (cost_euclid_squared)>
2026-06-15 17:11:52,359 uot.problems.iterator INFO: Generated problem <TwoMarginalProblem[GMM-2d-50pt] 5000x5000 with (cost_euclid_squared)>
2026-06-15 17:11:54,433 uot.problems.iterator INFO: Generated problem <TwoMarginalProblem[GMM-2d-50pt] 5000x5000 with (cost_euclid_squared)>
2026-06-15 17:11:54,433 uot.problems.iterator INFO: Generated problem <TwoMarginalProblem[GMM-2d-50pt] 5000x5000 with (cost_euclid_squared)>
2026-06-15 17:11:56,045 uot.problems.iterator INFO: Generated problem <TwoMarginalProblem[GMM-2d-50pt] 5000x5000 with (cost_euclid_squared)>
2026-06-15 17:11:56,045 uot.problems.iterator INFO: Generated problem <TwoMarginalProblem[GMM-2d-50pt] 5000x5000 with (cost_euclid_squared)>
2026-06-15 17:11:57,696 uot.problems.iterator INFO: Generated problem <TwoMarginalProblem[GMM-2d-50pt] 5000x5000 with (cost_euclid_squared)>
2026-06-15 17

,name,epsilon,cost,iterations,converged,time
0,ott-gw,0.5,0.021254293906562083,5,True,1506.644375
1,ott-gw,0.5,0.01802089698143894,5,True,1045.184875
2,ott-gw,0.5,0.0021176322206538645,5,True,1083.679417
3,ott-gw,0.5,0.0067128098419473314,5,True,1072.767292
4,ott-gw,0.5,0.018230995485072343,5,True,1055.665292
5,ott-gw,0.5,0.03655943895372982,5,True,1034.108209
6,ott-gw,0.5,0.004357257504149337,5,True,1049.873000
7,ott-gw,0.5,0.0032206234082039487,5,True,1047.710375
8,ott-gw,0.5,0.021711530776538868,5,True,1036.004583
9,ott-gw,0.5,0.007618989887942851,5,True,1050.595250


In [12]:
gw_ok_plot = gw_ok.copy()
for col in ['cost', 'time', 'iterations', 'error']:
    if col in gw_ok_plot.columns:
        gw_ok_plot[col] = gw_ok_plot[col].apply(lambda v: float(v) if v is not None else float('nan'))
gw_ok_plot['epsilon'] = gw_ok_plot['epsilon'].astype(float)
gw_grouped = gw_ok_plot.groupby('epsilon')[['cost', 'time', 'iterations']].median().reset_index().sort_values('epsilon')

fig_gw = make_subplots(rows=1, cols=3, subplot_titles=['Median GW cost', 'Median wall time (ms)', 'Median outer iterations'])
gw_color = solver_color('ott-gw')
kw = dict(x=gw_grouped['epsilon'], mode='lines+markers', name='ott-gw',
          line_color=gw_color, marker_color=gw_color)
fig_gw.add_trace(go.Scatter(**kw, y=gw_grouped['cost']), row=1, col=1)
fig_gw.add_trace(go.Scatter(**kw, y=gw_grouped['time'], showlegend=False), row=1, col=2)
fig_gw.add_trace(go.Scatter(**kw, y=gw_grouped['iterations'], showlegend=False), row=1, col=3)
fig_gw.update_xaxes(type='log', title_text='ε')
fig_gw.update_layout(height=400, width=1000, title='OTT Gromov–Wasserstein — median over problems')
fig_gw.show()

## 5  Sinkhorn divergence

The Sinkhorn divergence $S_\varepsilon(\mu,\nu) = \mathrm{OT}_\varepsilon(\mu,\nu) - \frac{1}{2}(\mathrm{OT}_\varepsilon(\mu,\mu) + \mathrm{OT}_\varepsilon(\nu,\nu))$ is a debiased version of the regularised OT cost. Here we run the OTT implementation across an epsilon grid.

In [13]:
# `sinkhorn_divergence_with_solver` is a plain function (not a BaseSolver subclass)
# so only the OTT variant runs through run_pipeline here.

eps_sd = [1.0, 0.5, 0.1]

solvers_sd = [
    SolverConfig(
        name='ott-sinkhorn-divergence',
        solver=OTTSinkhornDivergence,
        param_grid=[{'epsilon': eps, 'max_iterations': 2000, 'threshold': 1e-5}
                    for eps in eps_sd],
        is_jit=False,
        use_cost_matrix=False,
    ),
]

results_sd = run_pipeline(
    experiment=experiment,
    solvers=solvers_sd,
    iterators=[make_iterator()],
    folds=1,
    progress=False,
)

sd_ok = results_sd[results_sd['status'] == 'success'].copy()
sd_ok['eps'] = sd_ok['epsilon']
sd_ok[['name', 'eps', 'cost', 'converged', 'time']].head(12)

2026-06-15 17:12:42,340 uot.problems.iterator INFO: Generated problem <TwoMarginalProblem[GMM-2d-50pt] 5000x5000 with (cost_euclid_squared)>
2026-06-15 17:12:42,340 uot.problems.iterator INFO: Generated problem <TwoMarginalProblem[GMM-2d-50pt] 5000x5000 with (cost_euclid_squared)>
2026-06-15 17:12:43,781 uot.problems.iterator INFO: Generated problem <TwoMarginalProblem[GMM-2d-50pt] 5000x5000 with (cost_euclid_squared)>
2026-06-15 17:12:43,781 uot.problems.iterator INFO: Generated problem <TwoMarginalProblem[GMM-2d-50pt] 5000x5000 with (cost_euclid_squared)>
2026-06-15 17:12:45,252 uot.problems.iterator INFO: Generated problem <TwoMarginalProblem[GMM-2d-50pt] 5000x5000 with (cost_euclid_squared)>
2026-06-15 17:12:45,252 uot.problems.iterator INFO: Generated problem <TwoMarginalProblem[GMM-2d-50pt] 5000x5000 with (cost_euclid_squared)>
2026-06-15 17:12:46,650 uot.problems.iterator INFO: Generated problem <TwoMarginalProblem[GMM-2d-50pt] 5000x5000 with (cost_euclid_squared)>
2026-06-15 17

,name,eps,cost,converged,time
0,ott-sinkhorn-divergence,1.0,0.12985529196324067,True,858.134125
1,ott-sinkhorn-divergence,1.0,0.17338933895124656,True,884.901625
2,ott-sinkhorn-divergence,1.0,0.08053886736823784,True,820.387500
3,ott-sinkhorn-divergence,1.0,0.04670933767669674,True,833.805959
4,ott-sinkhorn-divergence,1.0,0.002345390275729803,True,846.801542
5,ott-sinkhorn-divergence,1.0,0.10554993649320948,True,854.274208
6,ott-sinkhorn-divergence,1.0,0.0006067572912285546,True,815.251625
7,ott-sinkhorn-divergence,1.0,0.019605943274872396,True,826.754333
8,ott-sinkhorn-divergence,1.0,0.0925594783272121,True,845.959750
9,ott-sinkhorn-divergence,1.0,0.09575071932964116,True,863.543417


In [14]:
sd_plot = sd_ok.copy()
for col in ['cost', 'time', 'error']:
    if col in sd_plot.columns:
        sd_plot[col] = sd_plot[col].apply(lambda v: float(v) if v is not None else float('nan'))
sd_plot['eps'] = sd_plot['eps'].astype(float)
sd_grouped = sd_plot.groupby(['name', 'eps'])['cost'].median().reset_index()

fig_sd = go.Figure()
for solver_name, grp in sd_grouped.groupby('name'):
    grp = grp.sort_values('eps')
    fig_sd.add_trace(go.Scatter(x=grp['eps'], y=grp['cost'], name=solver_name, mode='lines+markers',
                             line_color=solver_color(solver_name), marker_color=solver_color(solver_name)))

fig_sd.update_xaxes(type='log', title_text='ε')
fig_sd.update_yaxes(title_text='S_ε(μ, ν)  (median)')
fig_sd.update_layout(height=400, width=700, title='OTT Sinkhorn divergence — median over problems')
fig_sd.show()

## 6  Full comparison

Concatenate all results into one DataFrame so the Dash dashboard or any downstream analysis can consume them in a single go.

In [15]:
all_results = pd.concat(
    [results_sinkhorn, results_lr, results_gw, results_sd],
    ignore_index=True,
    sort=False,
)

print('Total rows:', len(all_results))
print('Solvers:', all_results['name'].unique().tolist())
print('Status counts:')
print(all_results['status'].value_counts())

all_results.head(5)

Total rows: 160
Solvers: ['native-sinkhorn-log', 'ott-sinkhorn', 'ott-lr-sinkhorn', 'ott-sinkhorn-ref', 'ott-gw', 'ott-sinkhorn-divergence']
Status counts:
status
success    160
Name: count, dtype: int64


,dataset,type,n_mu,n_nu,cost,time,transport_plan,iterations,error,status,...,epsilon,max_iterations,threshold,low_rank_plan,rank,gamma,gamma_rescale,linear_solver_lse_mode,linear_solver_max_iterations,linear_solver_threshold
0,GMM-2d-50pt,two_marginal,2500,2500,0.19166869967483038,399.173000,"[[1.5311821861929407e-43, 3.78379481452487e-43...",11,5.665016182913098e-17,success,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,GMM-2d-50pt,two_marginal,2500,2500,0.23974501828267356,395.126541,"[[2.870864552853671e-57, 8.047093743735687e-56...",11,5.2098645576808126e-17,success,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,GMM-2d-50pt,two_marginal,2500,2500,0.10919977796316198,389.336916,"[[7.0110029589519135e-87, 1.2658005125365223e-...",11,6.18696931224627e-17,success,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,GMM-2d-50pt,two_marginal,2500,2500,0.10420038392812925,427.939583,"[[6.639704751355308e-57, 4.440726118629697e-55...",11,6.766101306077088e-17,success,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,GMM-2d-50pt,two_marginal,2500,2500,0.08858915180502061,289.386417,"[[1.3000097927544238e-42, 1.0345427844238018e-...",11,5.3063670974221584e-17,success,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
_sdf = all_results[all_results['status'] == 'success'].copy()
for col in ['cost', 'time', 'iterations', 'error']:
    if col in _sdf.columns:
        _sdf[col] = _sdf[col].apply(lambda v: float(v) if v is not None else float('nan'))
_sdf['converged'] = _sdf['converged'].apply(lambda v: float(bool(v)) if v is not None else float('nan'))
summary = (
    _sdf
    .groupby('name')[['cost', 'time', 'iterations', 'converged']]
    .agg({
        'cost': ['median', 'std'],
        'time': ['median', 'std'],
        'iterations': 'median',
        'converged': 'mean',
    })
    .round(4)
)
summary.columns = ['_'.join(c).strip('_') for c in summary.columns]
summary

,cost_median,cost_std,time_median,time_std,iterations_median,converged_mean
name,,,,,,
native-sinkhorn-log,0.1120,0.0660,392.2317,1875.1828,11.0,1.0
ott-gw,0.0128,0.0107,1057.2961,90.6102,5.0,1.0
ott-lr-sinkhorn,0.1064,0.0642,892.2057,593.0984,50.0,1.0
ott-sinkhorn,0.1120,0.0660,345.5064,604.1694,10.0,1.0
ott-sinkhorn-divergence,0.0872,0.0570,858.5510,38.5965,50.0,1.0
ott-sinkhorn-ref,0.1194,0.0651,367.0570,19.0280,10.0,1.0


In [17]:
success = all_results[all_results['status'] == 'success'].copy()
for col in ['cost', 'time']:
    if col in success.columns:
        success[col] = success[col].apply(lambda v: float(v) if v is not None else float('nan'))

fig_violin = go.Figure()
for solver_name, grp in success.groupby('name'):
    valid_time = grp['time'].dropna()
    if valid_time.empty:
        continue
    fig_violin.add_trace(go.Violin(
        y=valid_time,
        name=solver_name,
        line_color=solver_color(solver_name),
        fillcolor=solver_color(solver_name),
        box_visible=True,
        meanline_visible=True,
        points='all',
        opacity=0.6,
    ))

fig_violin.update_yaxes(title_text='Wall time (ms)')
fig_violin.update_layout(
    height=500,
    width=1000,
    title='Wall-time distribution across all solvers and epsilon settings',
)
fig_violin.show()

In [18]:
# Cost vs time scatter — coloured by solver name.
fig_scatter = go.Figure()
for solver_name, grp in success.groupby('name'):
    grp_valid = grp[grp['cost'].notna() & grp['time'].notna()]
    if grp_valid.empty:
        continue
    fig_scatter.add_trace(go.Scatter(
        x=grp_valid['time'],
        y=grp_valid['cost'].apply(lambda v: float(v) if not isinstance(v, float) else v),
        mode='markers',
        name=solver_name,
        opacity=0.7,
        marker=dict(size=7, color=solver_color(solver_name)),
    ))

fig_scatter.update_xaxes(title_text='Wall time (ms)')
fig_scatter.update_yaxes(title_text='Transport cost')
fig_scatter.update_layout(
    height=500,
    width=900,
    title='Cost vs time per problem instance',
)
fig_scatter.show()

## Extras

### Using `to_ott_linear_problem` / `to_ott_quadratic_problem` directly

If you want to use OTT-JAX directly (without the uot-bench harness), the `TwoMarginalProblem` helpers give you first-class OTT objects.

In [19]:
from ott.solvers.linear.sinkhorn import Sinkhorn as OTTSinkhorn
from ott.solvers.quadratic.gromov_wasserstein import GromovWasserstein

prob = next(GaussianMixtureGenerator(**gen_cfg).generate())

# ── Linear OT ──────────────────────────────────────────────────────────────
lin_prob = prob.to_ott_linear_problem(epsilon=0.1)
lin_out  = OTTSinkhorn(max_iterations=1000)(lin_prob)
print('Direct OTT Sinkhorn: primal_cost =', float(lin_out.primal_cost), ' converged =', bool(lin_out.converged))

# ── Quadratic OT (GW) ──────────────────────────────────────────────────────
from ott.solvers.linear.sinkhorn import Sinkhorn as _S
quad_prob = prob.to_ott_quadratic_problem(epsilon=0.1)
gw_solver = GromovWasserstein(linear_solver=_S(max_iterations=200), epsilon=0.1)
gw_raw    = gw_solver(quad_prob)
# OTT pads gw_raw.costs with -1.0 for outer iterations that never ran, so take
# the last non-sentinel entry (this is what the OTTGromovWassersteinSolver wrapper does).
gw_costs = np.asarray(gw_raw.costs)
gw_costs = gw_costs[gw_costs != -1]
print('Direct OTT GW: last GW cost =', float(gw_costs[-1]), ' converged =', bool(gw_raw.converged))

# ── PointCloud geometry helper on a measure ────────────────────────────────
mu_meas = prob.get_marginals()[0]
geom = mu_meas.to_ott_geometry(prob.get_marginals()[1], cost_name='cost_euclid_squared', epsilon=0.1)
print('PointCloud geom:', geom, '  shape:', geom.shape)

Direct OTT Sinkhorn: primal_cost = 0.18306115075314405  converged = True
Direct OTT GW: last GW cost = 0.021248951150394536  converged = True
PointCloud geom: <ott.geometry.pointcloud.PointCloud object at 0x69187e2b0>   shape: (2500, 2500)


### Unbalanced OT via `tau_a` / `tau_b`

OTT-JAX supports unbalanced OT natively through `tau_a < 1` / `tau_b < 1`. These live on the OTT `LinearProblem`, so pass them to `to_ott_linear_problem(...)` (or, when using `run_pipeline`, put them in the `SolverConfig.param_grid` — the representation builder threads them into the `LinearProblem`).

In [20]:
one_problem = next(GaussianMixtureGenerator(**gen_cfg).generate())

# tau_a / tau_b live on the OTT LinearProblem, so build the problem with them.
# (Via run_pipeline you'd instead put tau_a/tau_b in the SolverConfig param_grid,
#  which the representation builder threads into the LinearProblem.)
lin_balanced   = one_problem.to_ott_linear_problem(epsilon=0.1, tau_a=1.0, tau_b=1.0)
lin_unbalanced = one_problem.to_ott_linear_problem(epsilon=0.1, tau_a=0.8, tau_b=0.8)

solver = OTTSinkhornSolver(max_iterations=2000)
bal_out   = solver.solve(lin_balanced,   epsilon=0.1)
unbal_out = solver.solve(lin_unbalanced, epsilon=0.1)

print(f'Balanced OT   cost = {float(bal_out["cost"]):.6f}')
print(f'Unbalanced OT cost = {float(unbal_out["cost"]):.6f}  (marginal constraints relaxed)')

Balanced OT   cost = 0.183061
Unbalanced OT cost = 0.430197  (marginal constraints relaxed)


### Save results to CSV (for the Dash dashboard)

The DataFrame has the same schema as results produced by the SLURM `uot-benchmark` CLI, so you can load it directly into the dashboard.

In [21]:
out_path = '../outputs/ott_interop_results.csv'
os.makedirs(os.path.dirname(out_path), exist_ok=True)

# Drop non-serialisable columns (JAX arrays, tuples) before saving.
serialisable = [
    c for c in all_results.columns
    if all_results[c].dtype in (float, int, bool, object)
    and c not in ('transport_plan', 'coupling', 'potentials',
                  'u_final', 'v_final', 'low_rank_plan')
]
all_results[serialisable].to_csv(out_path, index=False)
print('Saved to', out_path)

Saved to ../outputs/ott_interop_results.csv
